# Lab 12 — Online LLM API + Gradio
Run llama-server inside Colab, call it through an OpenAI-compatible API, and optionally expose a Gradio demo. Educational use only.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y cmake build-essential git
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp.git
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --target llama-server -j 1
!pip -q install huggingface_hub openai gradio requests

In [ ]:
from huggingface_hub import hf_hub_download
model_path=hf_hub_download(repo_id='lmstudio-community/SmolLM2-360M-Instruct-GGUF',filename='SmolLM2-360M-Instruct-Q4_K_M.gguf')
import subprocess,time,requests,os,signal
server=subprocess.Popen(['./llama.cpp/build/bin/llama-server','-m',model_path,'--host','127.0.0.1','--port','8080','-c','512'],stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
for _ in range(60):
 try:
  if requests.get('http://127.0.0.1:8080/health',timeout=2).ok: break
 except: pass
 time.sleep(1)
print('server ready')

In [ ]:
from openai import OpenAI
client=OpenAI(base_url='http://127.0.0.1:8080/v1',api_key='local')
models=requests.get('http://127.0.0.1:8080/v1/models',timeout=5).json().get('data',[])
model_id=models[0]['id'] if models else 'local'
resp=client.chat.completions.create(model=model_id,messages=[{'role':'system','content':'Educational healthcare tutor; no diagnosis or treatment.'},{'role':'user','content':'Explain specificity in simple language.'}],max_tokens=120,temperature=.2)
print(resp.choices[0].message.content)

In [ ]:
import gradio as gr
def ask(q):
 r=client.chat.completions.create(model=model_id,messages=[{'role':'system','content':'Educational healthcare tutor; no diagnosis, treatment or patient-specific advice.'},{'role':'user','content':q}],max_tokens=120,temperature=.2); return r.choices[0].message.content
LAUNCH=False
if LAUNCH: gr.Interface(ask,'text','text',title='Educational Healthcare LLM').launch(share=True)
else: print('Set LAUNCH=True to start Gradio')

In [ ]:
server.terminate(); print('server stopped')